# 모델 병합하기

LoRA 모델을 서빙하는 방법은 두 가지입니다.

- 기본 모델과 LoRA 어댑터의 새 가중치를 하나로 합친 병합(merged) 모델로 서빙
- 필요할 때 LoRA 어댑터와 기본 모델을 즉시 전환할 수 있도록 따로 서빙

두 번째 방식이 더 유연합니다. 여러 어댑터를 서로, 그리고 기본 모델과 쉽게 A/B 테스트할 수 있고, 사실상 여러 개의 서로 다른 모델을 매우 적은 자원으로 서빙할 수 있습니다.  
그렇지만 여기서는 단순함을 위해 병합 방식을 사용합니다.

CPU에서 실행하기 때문에 여기서는 오래 학습할 수 없었으므로, 조금 더 오래 학습한 가중치를 사용합니다(10 epoch, 이 워크벤치의 자원으로는 약 1.5시간이 걸립니다).  
이 가중치는 `socratic-tutor-lora-pre-trained` 폴더에 있습니다.

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [ ]:
BASE_MODEL = "Qwen/Qwen2-0.5B-Instruct"
LORA_DIR = "./socratic-tutor-lora-pre-trained"
OUT_DIR = "./socratic-tutor-lora-merged"

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# dtype 선택. 지원된다면 bf16이 좋고, 그렇지 않으면 fp16을 사용합니다.
# CPU만 있다면 float32를 사용합니다(더 느리고 크기도 커짐).
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
if not torch.cuda.is_available():
    dtype = torch.float32

# 이전 노트북에서 만든 tokenizer가 이미 있는지 확인
if 'tokenizer' in globals():
    print("✅ Reusing tokenizer from previous notebook")
else:
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# 이전 노트북(예: 평가 노트북)에서 만든 model이 이미 있는지 확인
if 'model' in globals() and hasattr(model, 'merge_and_unload'):
    print("✅ Reusing model from previous notebook (saving memory and time!)")
    # 모델은 이미 어댑터가 로드된 PeftModel임
    # 올바른 어댑터(저장하려는 사전 학습된 어댑터)인지 확인
    print("   Using existing PEFT model with adapter")
    peft_model = model
else:
    print("Loading base model...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=dtype,
        trust_remote_code=True,
    )
    
    print("Loading LoRA adapter...")
    peft_model = PeftModel.from_pretrained(base, LORA_DIR)

# LoRA 가중치를 기본 가중치에 병합하고 어댑터 모듈 제거
print("Merging adapter with base model...")
merged = peft_model.merge_and_unload()

# (선택 사항) 저장 전에 가중치가 연속 메모리에 있도록 보장하기도 함
merged = merged.to(dtype)

# 병합된 모델과 tokenizer 저장
print(f"Saving to {OUT_DIR}...")
merged.save_pretrained(OUT_DIR, safe_serialization=True)  # 가능하면 .safetensors로 저장
tokenizer.save_pretrained(OUT_DIR)

print(f"✅ Merged model saved to: {OUT_DIR}")

# 새 모델 저장하기

모델이 기대한 대로 작동하는 것을 확인했다면 저장해 보겠습니다!  
OpenShift 이미지 레지스트리에 modelcar로 저장하고, 메타데이터를 모델 레지스트리에 등록합니다.

In [ ]:
from model_registry import ModelRegistry
from model_registry.utils import OCIParams

In [ ]:
username = "<USER_NAME>"
cluster_domain = "<CLUSTER_DOMAIN>"
version = "0.0.1"

model_registry_url = f"https://{username}-registry-rest.{cluster_domain}"
print(f"registering to {model_registry_url}")

In [ ]:
%%bash
oc registry login \
  --to=/tmp/ocp-auth.json

In [ ]:
%env REGISTRY_AUTH_FILE=/tmp/ocp-auth.json

In [ ]:
from datetime import datetime

# 모델 메타데이터와 설명
model_description = """
Socratic Math Tutor - Fine-tuned Qwen2-0.5B model using LoRA.

This model is trained to act as a Socratic tutor for mathematics education. Instead of 
providing direct answers, it guides students through problem-solving by asking thoughtful 
questions that encourage critical thinking and discovery.

Training Details:
- Base Model: Qwen/Qwen2-0.5B-Instruct
- Fine-tuning Method: LoRA (Low-Rank Adaptation)
- Training Data: 30 curated Socratic tutoring examples (3x augmented = 90 samples)
- Training Epochs: 10
- LoRA Rank: 8
- Target Modules: q_proj, k_proj, v_proj, o_proj

Use Case:
- Educational applications requiring guided learning
- Math tutoring systems that emphasize critical thinking
- Interactive problem-solving assistants

Behavioral Characteristics:
- Responds with guiding questions rather than direct answers
- Encourages student exploration and discovery
- Maintains patient and supportive tone
"""

# 추적용 사용자 정의 속성
custom_properties = {
    "base_model": BASE_MODEL,
    "training_method": "LoRA",
    "lora_rank": "8",
    "lora_alpha": "16",
    "training_epochs": "10",
    "training_samples": "90",
    "dataset": "socratic-training-data-curated.jsonl",
    "use_case": "socratic_math_tutor",
    "training_date": datetime.now().strftime("%Y-%m-%d"),
    "merged": "true",
    "dtype": str(dtype),
}

oci_upload_params = OCIParams(
    base_image="busybox",
    oci_ref=f"default-route-openshift-image-registry.{cluster_domain}/{username}-canopy/socratic-model:{version}"
)

mr = ModelRegistry(model_registry_url, author=username)

registered_model = mr.upload_artifact_and_register_model(
    name="socratic-model",
    model_files_path=OUT_DIR,
    author=username,
    version=version,
    description=model_description,
    upload_params=oci_upload_params,
    model_format_name="vllm",
    model_format_version="1",
    metadata=custom_properties,
)

print(f"✅ Model registered successfully!")
print(f"   Name: {registered_model.name}")
print(f"   Version: {version}")
print(f"   Author: {username}")

모델을 저장하고 등록했으니 다음 장인 **Deploy To Canopy**로 넘어갈 수 있습니다 🚀